🌟 DeepLoc 2.1 Pro 버전 - Google Drive + FP16 + CNN (발표용)

1. 드라이브 마운트

In [2]:
from google.colab import drive
drive.mount('/content/drive')  # 드라이브 마운트

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. 환경 최적화

In [3]:
!apt-get update -qq && apt-get install -y cd-hit
!pip install transformers torch scikit-learn pandas requests tqdm biopython h5py gdown -q

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import h5py
import pandas as pd
import numpy as np
from google.colab import drive
from transformers import EsmTokenizer, EsmModel
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tqdm import tqdm
import os
from Bio import SeqIO

drive.mount('/content/drive')
SAVE_PATH = '/content/drive/MyDrive/DeepLoc_Pro/'
os.makedirs(SAVE_PATH, exist_ok=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💾 Drive: {SAVE_PATH} | GPU: {torch.cuda.is_available()}")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  cd-hit
0 upgraded, 1 newly installed, 0 to remove and 78 not upgraded.
Need to get 521 kB of archives.
After this operation, 1,082 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 cd-hit amd64 4.8.1-4 [521 kB]
Fetched 521 kB in 2s (315 kB/s)
Selecting previously unselected package cd-hit.
(Reading database ... 121689 files and directories currently installed.)
Preparing to unpack .../cd-hit_4.8.1-4_amd64.deb ...
Unpacking cd-hit (4.8.1-4) ...
Setting up cd-hit (4.8.1-4) ...
Processing triggers for man-db (2.10.2-1) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.7 MB/s eta 0:00:00
Drive alrea

In [ ]:
import requests
import time
from io import StringIO

def prepare_deeploc_pro(num_samples=1000):
    """DeepLoc 2.1 데이터셋 (필드명 수정 완료)"""
    h5_path = SAVE_PATH + 'deeploc_raw.h5'

    if os.path.exists(h5_path):
        print("✅ Drive에서 로드")
        with h5py.File(h5_path, 'r') as f:
            df = pd.DataFrame({
                'entry_id': [x.decode('utf-8') if isinstance(x, bytes) else x for x in f['ids'][:]],
                'sequence': [x.decode('utf-8') if isinstance(x, bytes) else x for x in f['seqs'][:]],
                'cc_subcell': [x.decode('utf-8') if isinstance(x, bytes) else x for x in f['annotations'][:]]
            })
        print(f"📊 로드: {len(df)}개")
    else:
        print("📥 UniProt 수집 중...")
        # 🔑 정정: cc_subcell (cc_subcell_location 아님!)
        query = "(taxonomy_id:2759) AND (reviewed:true) AND (cc_subcell:*)"

        params = {'query': query, 'format': 'fasta'}
        r = requests.get("https://rest.uniprot.org/uniprotkb/stream", params=params, timeout=30)

        if r.status_code == 200 and len(r.text) > 100:
            seqs = list(SeqIO.parse(StringIO(r.text), 'fasta'))
            df = pd.DataFrame({
                'entry_id': [s.id.split('|')[1] for s in seqs],
                'sequence': [str(s.seq) for s in seqs],
                'cc_subcell': ['Eukaryota'] * len(seqs)  # 라벨
            })
            print(f"✅ 수집: {len(df)}개")
        else:
            raise ValueError(f"API {r.status_code}: {r.text[:100]}")

        # HDF5 저장
        with h5py.File(h5_path, 'w') as f:
            f.create_dataset('ids', data=df['entry_id'].values.astype('S'))
            f.create_dataset('seqs', data=df['sequence'].values.astype('S'))
            f.create_dataset('annotations', data=df['cc_subcell'].values.astype('S'))
        print(f"💾 저장: {len(df)}개")

    # CD-HIT 40%
    print("\n🔬 CD-HIT...")
    fasta_path = SAVE_PATH + 'temp_raw.fasta'
    with open(fasta_path, 'w') as f:
        for i, row in df.iterrows():
            f.write(f">{row['entry_id']}\n{row['sequence']}\n")

    nr_path = SAVE_PATH + 'deeploc_nr40.fasta'
    os.system(f"cd-hit -i {fasta_path} -o {nr_path} -c 0.4 -n 2 -M 0 -T 2 > /dev/null 2>&1")

    nr_records = list(SeqIO.parse(nr_path, 'fasta'))
    nr_df = pd.DataFrame([{'entry_id': rec.id, 'sequence': str(rec.seq), 'length': len(rec.seq)} for rec in nr_records])
    nr_df.to_csv(SAVE_PATH + 'deeploc_nr40.csv', index=False)
    print(f"✅ 완료: {len(nr_df)}개 NR 서열")

    return nr_df

# 실행!
df_final = prepare_deeploc_pro()
print(df_final.head())

📥 UniProt 수집 중...


ValueError: API 400: Error messages
'cc_subcell' is not a valid search field

In [ ]:
# ===== 3. 논문 큐레이션 (Supplementary Table 2 정확 재현) =====
def deeploc_curation(df):
    """이미지 테이블 모든 단계"""
    print(f"🔧 큐레이션 시작: {len(df):,}")

    # Step 1: Isoform & other removal (28,417 → 28,030)
    bad_terms = ['isoform', 'cleav', 'secreted form', 'process', 'shed']
    def clean_labels(text):
        if pd.isna(text): return []
        labels = text.split('; ')
        return [l for l in labels if not any(term in l.lower() for term in bad_terms)]

    df['clean_cc'] = df['cc_subcell_location'].apply(clean_labels)
    df = df[df['clean_cc'].str.len() > 0]
    print(f"📉 Isoform removal: {len(df):,}")

    # Step 2: Ambiguous AA (27,461)
    df = df[~df['sequence'].str.contains('[BUZX]', regex=True, na=False)]
    print(f"🧬 Ambiguous AA removal: {len(df):,}")

    # Step 3: Multi-chain heuristic (27,525 추정)
    def has_multichain(text):
        return bool(re.search(r'(CHAIN|Fragment).*?-?\d+', str(text), re.I))
    df = df[~df['cc_subcell_location'].apply(has_multichain)]
    print(f"🔗 Multi-chain removal: {len(df):,}")

    # Step 4: Final Eukaryotic (25,240 목표)
    df = df[df['taxonomy'].str.contains('Eukaryota', na=False)]
    print(f"🌱 Final Eukaryotic: {len(df):,} ✅")
    return df

df_final = deeploc_curation(df_raw)

In [ ]:
# ===== 4. 라벨 추출 (10 위치 + 4 막 타입) =====
LOCATIONS = ['Nucleus', 'Cytoplasm', 'Mitochondrion', 'Plasma membrane', 'Endoplasmic reticulum',
             'Golgi apparatus', 'Lysosome', 'Peroxisome', 'Extracellular']
MEMBRANE_TYPES = ['Peripheral membrane', 'Transmembrane', 'Lipid-anchored', 'Soluble']

def extract_multilabel(text_list):
    label_dict = {f'loc_{l.replace(" ", "_")}': 0 for l in LOCATIONS}
    label_dict.update({f'mem_{m.replace(" ", "_")}': 0 for m in MEMBRANE_TYPES})

    if isinstance(text_list, list):
        text = ' '.join(text_list).lower()
        for loc in LOCATIONS:
            if loc.lower() in text: label_dict[f'loc_{loc.replace(" ", "_")}'] = 1
        if 'peripheral membrane' in text: label_dict['mem_Peripheral_membrane'] = 1
        elif 'transmembrane' in text: label_dict['mem_Transmembrane'] = 1
        elif 'lipid-anchored' in text: label_dict['mem_Lipid-anchored'] = 1
        else: label_dict['mem_Soluble'] = 1
    return pd.Series(label_dict)

labels_df = df_final['clean_cc'].apply(extract_multilabel)
df_dataset = pd.concat([df_final[['entry_id', 'sequence']], labels_df], axis=1)
df_dataset.to_csv('deeploc_25k.csv', index=False)
print("💾 deeploc_25k.csv 저장 완료!")



In [ ]:
# ===== 5. ESM-1b + ProtT5 임베딩 추출 (DeepLoc 핵심) =====
print("\n🤖 ESM-1b 임베딩 추출 중...")
esm_model, esm_alphabet = esm.pretrained.esm1b_t33_650M_UR50S_1()
esm_model = esm_model.to(device)
esm_batch_converter = esm_alphabet.get_batch_converter()

prot_t5_tokenizer = AutoTokenizer.from_pretrained("Rostlab/prot_t5_xl_uniref50", do_lower_case=False)
prot_t5_model = AutoModel.from_pretrained("Rostlab/prot_t5_xl_uniref50-enc").to(device)

def extract_esm_embeddings(sequences, model=esm_model, converter=esm_batch_converter, device=device, max_len=1022):
    data = [("protein", s[:max_len]) for s in sequences]
    batch_labels, batch_strs, batch_tokens = converter(data)
    batch_tokens = batch_tokens.to(device)

    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=False)
    embeddings = results["representations"][33]  # [batch, seq_len, 1280]
    return embeddings.mean(1).cpu().numpy()  # per-protein 평균

def extract_prot_t5_embeddings(sequences, tokenizer=prot_t5_tokenizer, model=prot_t5_model, device=device):
    embeddings = []
    for seq in tqdm(sequences, desc="ProtT5"):
        inputs = tokenizer(seq, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)
        with torch.no_grad():
            outputs = model(**inputs)
        emb = outputs.last_hidden_state.mean(1).cpu().numpy()
        embeddings.append(emb)
    return np.vstack(embeddings)

# 배치 임베딩 (메모리 최적화)
batch_size = 8
esm_embs = []
prot5_embs = []

for i in tqdm(range(0, len(df_dataset), batch_size), desc="임베딩 생성"):
    batch_seqs = df_dataset['sequence'].iloc[i:i+batch_size].tolist()

    # ESM
    esm_embs.append(extract_esm_embeddings(batch_seqs))

    # ProtT5 (느림 - 샘플링 권장)
    if i < 1000:  # Colab 테스트용 1k개 제한
        prot5_embs.append(extract_prot_t5_embeddings(batch_seqs))

    torch.cuda.empty_cache()

esm_embeddings = np.vstack(esm_embs)
print(f"✅ ESM 임베딩: {esm_embeddings.shape}")



In [ ]:
# ===== 6. DeepLoc 모델 훈련 (Attention pooling + MLP 대신 RF 간단 구현) =====
label_cols = [col for col in df_dataset.columns if col.startswith(('loc_', 'mem_'))]
X = esm_embeddings[:1000]  # 테스트용
y = df_dataset[label_cols].iloc[:1000].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = MultiOutputClassifier(RandomForestClassifier(n_estimators=100, random_state=42))
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("\n🏆 분류 성능:")
print(classification_report(y_test, y_pred, target_names=label_cols))

In [ ]:
# ===== 7. 최종 저장 =====
np.savez_compressed('deeploc_embeddings.npz', esm=esm_embeddings, prot_t5=np.array(prot5_embs))
print("\n🎉 COMPLETE! 파일 목록:")
print("- deeploc_25k.csv (25k 데이터셋)")
print("- deeploc_embeddings.npz (임베딩)")
print("- 모델 훈련 완료!")